# Validazione indipendente — mpmath e Hilbert

Fixture in ops/oracles/numerical-reference.json: mpmath 1.3.0, 200 cifre di lavoro e 160 cifre salvate. Il generatore non chiama MathemaSharp. Nei test xUnit sono confrontati gamma, log-gamma, chi-quadro e sistemi Hilbert. Python non serve per consumare le fixture salvate.

Questo notebook evidenzia perché un residuo piccolo non certifica una soluzione accurata. Il riferimento Hilbert usa elementi razionali e b=H·1 calcolati ad alta precisione, poi arrotondati nel tipo del consumatore: tale arrotondamento può perturbare molto la soluzione su matrici mal condizionate.


In [ ]:
#load "../shared/MathemaSharp.Numerics.csx"
#r "System.Text.Json"
using System;
using System.IO;
using System.Linq;
using System.Numerics;
using System.Text.Json;
using System.Globalization;
using MathemaSharp.TimeSeries;
using MathemaSharp.LinearAlgebra;
var root = new DirectoryInfo(Directory.GetCurrentDirectory());
while(root != null && !File.Exists(Path.Combine(root.FullName,"src","MathemaSharp.slnx"))) root=root.Parent;
if(root==null) throw new Exception("Run from repository root using the supplied runner");
var oracle=JsonDocument.Parse(File.ReadAllText(Path.Combine(root.FullName,"ops","oracles","numerical-reference.json")));
Console.WriteLine("oracle_precision: "+oracle.RootElement.GetProperty("precision_decimal_digits").GetInt32());
void CheckTails<T>() where T:IFloatingPointIeee754<T>
{
    var worst=T.Zero;
    var tolerance=T.Parse("1e-10",CultureInfo.InvariantCulture);
    foreach(var row in oracle.RootElement.GetProperty("gamma_q").EnumerateArray())
    {
        var a=T.Parse(row.GetProperty("a").GetString(),CultureInfo.InvariantCulture);
        var x=T.Parse(row.GetProperty("x").GetString(),CultureInfo.InvariantCulture);
        var expected=T.Parse(row.GetProperty("q").GetString(),CultureInfo.InvariantCulture);
        var actual=SpecialFunctions.RegularizedGammaQ(a,x);
        var error=T.Abs(actual/expected-T.One);
        if(!T.IsFinite(error)||error>tolerance) throw new Exception("Tail mismatch");
        worst=T.Max(worst,error);
    }
    Console.WriteLine(typeof(T).Name+"_tail_max_relative_error: "+worst.ToString(null,CultureInfo.InvariantCulture));
}
CheckTails<double>();
CheckTails<Decimal128>();


oracle_precision: 200
Double_tail_max_relative_error: 1.457167719820518E-12
Decimal128_tail_max_relative_error: 2.46279182553032713140E-13

In [ ]:
var h12=oracle.RootElement.GetProperty("hilbert").EnumerateArray().Single(row=>row.GetProperty("n").GetInt32()==12);
var rows=h12.GetProperty("matrix").EnumerateArray().ToArray();
var matrix=new double[12,12];
for(var i=0;i<12;i++){var values=rows[i].EnumerateArray().ToArray();for(var j=0;j<12;j++) matrix[i,j]=double.Parse(values[j].GetString(),CultureInfo.InvariantCulture);}
var rhs=h12.GetProperty("rhs").EnumerateArray().Select(value=>double.Parse(value.GetString(),CultureInfo.InvariantCulture)).ToArray();
var solution=LuFactorization<double>.Create(matrix).Solve(rhs);
var forwardError=solution.Max(value=>Math.Abs(value-1));
var residual=MatrixAlgebra.Multiply(matrix,solution).Zip(rhs,(actual,expected)=>actual-expected).ToArray();
var relativeResidual=MatrixAlgebra.NormL2<double>(residual)/MatrixAlgebra.NormL2<double>(rhs);
if(forwardError<=1e-4 || relativeResidual>=1e-12) throw new Exception("Hilbert limitation not reproduced");
Console.WriteLine("hilbert12_solution_error: "+forwardError.ToString("G17",CultureInfo.InvariantCulture));
Console.WriteLine("hilbert12_relative_residual: "+relativeResidual.ToString("G17",CultureInfo.InvariantCulture));
Console.WriteLine("small_residual_not_accuracy: OK");


hilbert12_solution_error: 0.0059542215231878082
hilbert12_relative_residual: 1.3119781642404253E-16
small_residual_not_accuracy: OK

## Interpretazione e limiti
Le fixture estreme gamma arrivano a probabilità dell'ordine di 1e-306: si verifica errore relativo, non una tolleranza assoluta che accetterebbe zero. I test float distinguono l'underflow del riferimento arrotondato; Half/Decimal32 non sono certificati su queste code. Lanczos resta a precisione limitata anche in Decimal128. Hilbert 3/5/8 è verificata in double e Decimal128; Hilbert12 double espone un limite di LU e viene rifiutata dalla QR attuale. Non è una promessa su tutte le matrici o tutti i domini.

Rigenerazione fixture su Windows: dotnet msbuild ops/numerical-oracles.proj -t:Setup,Generate. Dipendenza mpmath isolata in una virtualenv ignorata da Git. [Roadmap](../../ROADMAP.md), [Inventario](../../ARCHITECTURE.md).
